# Generate Quantized Averaged-Entropy Data

This Generate notebook computes and caches the data consumed by `Visualize-quantized-averaged-entropy.ipynb`. The companion Visualize notebook reproduces the coarse-grained state-distribution entropy table at trajectory length $T=1000$ and the empirical state-distribution entropy figure as a function of trajectory length for the numerical input families presented in **Main Numerical Input Suite and Next-State Extrapolation**. Both results use 100 equal-width bins with shared edges and summarize the same 20 realizations by their mean and sample standard deviation.


Set imports, ensemble constants, relative cache path, input labels, and dynamical-system parameters.


In [ ]:
from pathlib import Path

import numpy as np
from scipy.integrate import solve_ivp

# Ensemble, quantization, and entropy configuration
SEED = 42
N_REPLICATES = 20
N_STATES = 1000
N_BINS = 100
CONTEXT_LENGTHS = np.arange(50, N_STATES + 1, 50)

STATE_MIN = 50
STATE_MAX = 950
STATE_SPAN = STATE_MAX - STATE_MIN
HISTOGRAM_RANGE = (STATE_MIN - 0.5, STATE_MAX + 0.5)

# Relative cache path shared with the Visualize notebook
DATA_PATH = Path("quantized_averaged_entropy_100_bins.npz")

# Input-family order, table labels, and dynamical classes
DATA_NAMES = np.array([
    "Constant sampled",
    "Logistic r=3.2",
    "Logistic r=3.5",
    "Logistic r=3.55",
    "Logistic r=3.566",
    "Logistic r=3.5692",
    "Logistic r=3.5698",
    "Logistic r=3.9",
    "Logistic r=4.0",
    "Lorenz",
])

DISPLAY_NAMES = np.array([
    "Constant sampled",
    "Logistic r=3.2 (period 2)",
    "Logistic r=3.5 (period 4)",
    "Logistic r=3.55 (period 8)",
    "Logistic r=3.566 (period 16)",
    "Logistic r=3.5692 (period 32)",
    "Logistic r=3.5698 (period 64)",
    "Logistic r=3.9 (chaotic)",
    "Logistic r=4.0 (chaotic)",
    "Lorenz",
])

INPUT_CLASSES = np.array([
    "constant",
    "periodic", "periodic", "periodic", "periodic", "periodic", "periodic",
    "chaotic", "chaotic",
    "lorenz",
])

# Periodic and chaotic logistic-map regimes and burn-in controls
PERIODIC_LOGISTIC = (
    (3.2, 2),
    (3.5, 4),
    (3.55, 8),
    (3.566, 16),
    (3.5692, 32),
    (3.5698, 64),
)
CHAOTIC_LOGISTIC = (3.9, 4.0)
LOGISTIC_LONG_N = 20000
LOGISTIC_SETTLE_TOL = 1e-10

# Lorenz parameters and sampling grid
LORENZ_ARGS = (10.0, 28.0, 8.0 / 3.0)
LORENZ_T_SPAN = (0.0, 100.0)
LORENZ_T_EVAL = np.linspace(*LORENZ_T_SPAN, N_STATES)

print(f"Input types: {len(DATA_NAMES)}")
print(f"Replicates per input: {N_REPLICATES}")
print(f"States per replicate: {N_STATES}")
print(f"Histogram bins: {N_BINS}")


Define logistic-map and Lorenz dynamics, trajectory quantization, and base-2 entropy.


In [ ]:
def logistic_sequence(r, x0, n):
    values = np.empty(int(n), dtype=float)
    values[0] = float(x0)
    for index in range(1, int(n)):
        values[index] = float(r) * values[index - 1] * (1.0 - values[index - 1])
    return values


def settle_index(values, period, tolerance=LOGISTIC_SETTLE_TOL):
    # Find the first index after which lag-period error remains below tolerance
    error = np.abs(values[period:] - values[:-period])
    suffix_max = np.maximum.accumulate(error[::-1])[::-1]
    matches = np.flatnonzero(suffix_max < float(tolerance))
    return int(matches[0]) if matches.size else None


def periodic_logistic_window(r, x0, period, phase):
    # Discard transients and select a reproducibly phase-shifted periodic window
    long_trajectory = logistic_sequence(r, x0, LOGISTIC_LONG_N)
    start = settle_index(long_trajectory, period)
    if start is None:
        raise ValueError(f"No settled period-{period} orbit found for r={r}")
    start += int(phase) % int(period)
    stop = start + N_STATES
    if stop > long_trajectory.size:
        raise ValueError(f"The settled trajectory for r={r} is too short")
    return long_trajectory[start:stop].copy()


def lorenz_rhs(t, state, sigma, rho, beta):
    x, y, z = state
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]


def solve_lorenz_x(initial_conditions):
    solution = solve_ivp(
        lorenz_rhs,
        LORENZ_T_SPAN,
        initial_conditions,
        args=LORENZ_ARGS,
        t_eval=LORENZ_T_EVAL,
    )
    if not solution.success:
        raise RuntimeError(solution.message)
    return solution.y[0]


def quantize(values):
    # Scale each nonconstant trajectory independently into the 050..950 state range
    values = np.asarray(values, dtype=float)
    value_min = float(values.min())
    value_max = float(values.max())
    if value_max == value_min:
        raise ValueError("Constant trajectories require an explicit shared-scale token")
    scaled = STATE_MIN + (values - value_min) / (value_max - value_min) * STATE_SPAN
    return np.round(scaled).astype(np.int16)


def entropy(values):
    # Use shared histogram edges for every input family and realization
    counts, _ = np.histogram(values, bins=N_BINS, range=HISTOGRAM_RANGE)
    probabilities = counts[counts > 0].astype(float) / counts.sum()
    return float(-np.sum(probabilities * np.log2(probabilities)))


Build the constant, logistic, and Lorenz ensemble inputs.


In [ ]:
# Share each realization's sampled x0 across all input families
rng = np.random.default_rng(SEED)
phase_rng = np.random.default_rng(SEED + 1)
x0_samples = rng.uniform(0.3, 0.7, size=N_REPLICATES)

quantized = np.empty(
    (len(DATA_NAMES), N_REPLICATES, N_STATES),
    dtype=np.int16,
)

for replicate, x0 in enumerate(x0_samples):
    # Repeat one sampled token throughout each constant realization
    constant_token = np.round(STATE_MIN + STATE_SPAN * x0)
    quantized[0, replicate] = int(constant_token)

    # Use settled periodic windows with reproducibly randomized phase
    for input_index, (r, period) in enumerate(PERIODIC_LOGISTIC, start=1):
        phase = phase_rng.integers(0, period)
        values = periodic_logistic_window(r, x0, period, phase)
        quantized[input_index, replicate] = quantize(values)

    # Start chaotic trajectories directly from the shared sampled x0
    for input_index, r in enumerate(CHAOTIC_LOGISTIC, start=7):
        values = logistic_sequence(r, x0, N_STATES)
        quantized[input_index, replicate] = quantize(values)

    # Perturb Lorenz x0 and quantize the resulting x-coordinate
    lorenz_initial = [-8.0 + (x0 - 0.5), 7.0, 27.0]
    quantized[9, replicate] = quantize(solve_lorenz_x(lorenz_initial))

assert quantized.min() >= STATE_MIN
assert quantized.max() <= STATE_MAX
print(f"Generated {quantized.shape[0] * quantized.shape[1]} trajectories")


Evaluate entropy across trajectory lengths and save the visualization cache.


In [ ]:
# Retain one entropy per input family, realization, and trajectory length
entropy_per_replicate = np.empty(
    (len(DATA_NAMES), N_REPLICATES, len(CONTEXT_LENGTHS)),
    dtype=float,
)

for input_index in range(len(DATA_NAMES)):
    for replicate in range(N_REPLICATES):
        for length_index, length in enumerate(CONTEXT_LENGTHS):
            entropy_per_replicate[input_index, replicate, length_index] = entropy(
                quantized[input_index, replicate, :length]
            )

# Summarize curves across realizations and use T=1000 for the table
figure_entropy_mean = entropy_per_replicate.mean(axis=1)
figure_entropy_std = entropy_per_replicate.std(axis=1, ddof=1)
table_entropy_per_replicate = entropy_per_replicate[:, :, -1]
table_entropy_mean = table_entropy_per_replicate.mean(axis=1)
table_entropy_std = table_entropy_per_replicate.std(axis=1, ddof=1)

assert np.allclose(table_entropy_mean, figure_entropy_mean[:, -1])
assert np.allclose(table_entropy_std, figure_entropy_std[:, -1])

# Save compact table and figure-reproduction data with generation metadata
np.savez_compressed(
    DATA_PATH,
    schema_version=np.array(1, dtype=np.int64),
    seed=np.array(SEED, dtype=np.int64),
    n_replicates=np.array(N_REPLICATES, dtype=np.int64),
    n_states=np.array(N_STATES, dtype=np.int64),
    n_bins=np.array(N_BINS, dtype=np.int64),
    state_support=np.array([STATE_MIN, STATE_MAX], dtype=np.int64),
    histogram_range=np.array(HISTOGRAM_RANGE, dtype=float),
    data_names=DATA_NAMES,
    display_names=DISPLAY_NAMES,
    input_classes=INPUT_CLASSES,
    x0_samples=x0_samples,
    context_lengths=CONTEXT_LENGTHS,
    entropy_per_replicate=entropy_per_replicate,
    table_entropy_per_replicate=table_entropy_per_replicate,
    table_entropy_mean=table_entropy_mean,
    table_entropy_std=table_entropy_std,
    figure_entropy_mean=figure_entropy_mean,
    figure_entropy_std=figure_entropy_std,
)

print(f"Saved {DATA_PATH}")
print(f"Table rows: {len(DATA_NAMES)}")
print(f"Figure points per curve: {len(CONTEXT_LENGTHS)}")
